In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle
import io
from google.colab import files

print("Silakan upload file 'netflix_titles.csv' dari komputer/laptop:")
uploaded = files.upload()
nama_file = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[nama_file]))

print("\nData berhasil di-load! Dimensi:", df.shape)

In [ ]:
# 1. Penanganan Duplikat & Missing Value
df.drop_duplicates(inplace=True)
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])
df['country'] = df['country'].fillna(df['country'].mode()[0])
df['listed_in'] = df['listed_in'].fillna('Unknown')
df['type'] = df['type'].fillna('Movie')

# 2. FEATURE ENGINEERING: Membuat Target Y Baru (Kids vs Adults)
kids_ratings = ['TV-Y', 'TV-Y7', 'TV-Y7-FV', 'TV-G', 'PG', 'TV-PG', 'PG-13', 'G']
df['target_age'] = df['rating'].apply(lambda x: 0 if x in kids_ratings else 1)

le_type = LabelEncoder()
df['type_encoded'] = le_type.fit_transform(df['type'])

le_genre = LabelEncoder()
df['genre_encoded'] = le_genre.fit_transform(df['listed_in'])

le_country = LabelEncoder()
df['country_encoded'] = le_country.fit_transform(df['country'])

df['year_numeric'] = df['release_year']

Q1 = df['year_numeric'].quantile(0.25)
Q3 = df['year_numeric'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean = df[(df['year_numeric'] >= lower_bound) & (df['year_numeric'] <= upper_bound)].copy()
print(f"Jumlah baris setelah dibersihkan dari outlier: {len(df_clean)}\n")

plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.boxplot(data=df_clean, x='year_numeric', color='lightgreen')
plt.title('Boxplot Release Year (Bersih Outlier)')

plt.subplot(1, 3, 2)
sns.countplot(data=df_clean, x='target_age', palette='viridis')
plt.title('Distribusi Target (0: Kids/Teens, 1: Adults)')

plt.subplot(1, 3, 3)
fitur_korelasi = df_clean[['year_numeric', 'type_encoded', 'genre_encoded', 'country_encoded', 'target_age']]
sns.heatmap(fitur_korelasi.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix')

plt.tight_layout()
plt.show()

In [ ]:
X = df_clean[['year_numeric', 'type_encoded', 'genre_encoded', 'country_encoded']]
y = df_clean['target_age']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_dt = DecisionTreeClassifier(random_state=42, max_depth=8)
model_dt.fit(X_train_scaled, y_train)

model_rf = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=8)
model_rf.fit(X_train_scaled, y_train)

print("Training selesai! Model siap dievaluasi.")

In [ ]:
def tampilkan_evaluasi(model, X_test, y_test, nama_model):
    y_pred = model.predict(X_test)

    print(f"=== HASIL EVALUASI: {nama_model} ===")
    print(f"Akurasi  : {accuracy_score(y_test, y_pred)*100:.2f}%")
    print(f"Presisi  : {precision_score(y_test, y_pred, zero_division=0)*100:.2f}%")
    print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0)*100:.2f}%")
    print(f"F1-Score : {f1_score(y_test, y_pred, zero_division=0)*100:.2f}%")

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges')
    plt.title(f'Confusion Matrix - {nama_model}')
    plt.ylabel('Aktual (0:Kids, 1:Adults)')
    plt.xlabel('Prediksi')
    plt.show()

tampilkan_evaluasi(model_dt, X_test_scaled, y_test, "DECISION TREE")
tampilkan_evaluasi(model_rf, X_test_scaled, y_test, "RANDOM FOREST")

In [ ]:
with open('rf_age_model.pkl', 'wb') as file:
    pickle.dump(model_rf, file)

with open('scaler_age.pkl', 'wb') as file:
    pickle.dump(scaler, file)

with open('le_type.pkl', 'wb') as file:
    pickle.dump(le_type, file)

with open('le_genre.pkl', 'wb') as file:
    pickle.dump(le_genre, file)

with open('le_country.pkl', 'wb') as file:
    pickle.dump(le_country, file)

print("Semua file .pkl berhasil dibuat! Siap diunduh untuk website Streamlit.")